# Can Pickup Depth Demo

Today's test flow: manual can placement, optional hardcoded base move, arm pickup, DepthNet-based success confirmation. Object detection and path planning are intentionally mocked.

In [ ]:
import time
from pathlib import Path

from depth_camera import DepthCamera, mean_of_stats

# Main knobs for today's test.
DRY_RUN_ARM = False
ENABLE_BASE_MOVE = False
MONITOR_SECONDS = 30.0
MONITOR_INTERVAL_SECONDS = 1.0

# Region in the depth field to watch. Start with image center.
# Format: x1, y1, x2, y2 as normalized ratios.
DEPTH_REGION = (0.40, 0.40, 0.60, 0.60)

# Success heuristic: after pickup/lift, the watched region should change.
# Depth values are relative-ish, so tune this from logs.
SUCCESS_DEPTH_DELTA = 0.20

# Arm pose guesses. Keep these easy to edit in the notebook.
ARM_SPEED = 80
GRIPPER_SPEED = 120
SAFE_HOME_POSE = [(1, 0), (4, 0), (5, 0)]
PRE_GRASP_POSE = [(2, 18), (3, -12)]
REACH_POSE = [(2, 26), (3, -22)]
LIFT_POSE = [(2, 5), (3, 18)]
GRIPPER_OPEN = 0
GRIPPER_CLOSE_1 = -45
GRIPPER_CLOSE_2 = -55


In [ ]:
ttl_servo = None
robot = None

if not DRY_RUN_ARM:
    from SCSCtrl import TTLServo
    ttl_servo = TTLServo

if ENABLE_BASE_MOVE:
    from jetbot import Robot
    robot = Robot()

print('arm dry run:', DRY_RUN_ARM)
print('base move enabled:', ENABLE_BASE_MOVE)


In [ ]:
def move_servo(servo_id, angle, speed=ARM_SPEED, label=''):
    print('[arm] servo={} angle={} speed={} {}'.format(servo_id, angle, speed, label))
    if ttl_servo is not None:
        ttl_servo.servoAngleCtrl(servo_id, angle, 1, speed)
    time.sleep(0.25)

def apply_pose(name, pose, speed=ARM_SPEED):
    print('[arm] pose:', name)
    for servo_id, angle in pose:
        move_servo(servo_id, angle, speed, name)
    time.sleep(0.4)

def reset_arm():
    # Use only the safe measured home pose. Do not assume all-zero is safe.
    apply_pose('safe_home', SAFE_HOME_POSE, speed=100)

def open_gripper():
    move_servo(4, GRIPPER_OPEN, GRIPPER_SPEED, 'open_gripper')

def close_gripper():
    move_servo(4, GRIPPER_CLOSE_1, GRIPPER_SPEED, 'close_1')
    time.sleep(0.5)
    move_servo(4, GRIPPER_CLOSE_2, GRIPPER_SPEED, 'close_2')

def hardcoded_base_move():
    if robot is None:
        print('[base] skipped')
        return
    try:
        print('[base] hardcoded tiny move')
        robot.forward(0.12)
        time.sleep(0.25)
    finally:
        robot.stop()


In [ ]:
depth = DepthCamera(width=320, height=240, network='fcn-mobilenet')
depth.start(warmup_frames=2)


In [ ]:
def calibrate_depth_baseline(seconds=3.0):
    print('[calib] hold the target scene steady')
    stats = depth.observe_many(seconds=seconds, interval=0.5, region=DEPTH_REGION)
    baseline = mean_of_stats(stats, 'mean')
    print('[calib] frames={}, baseline_mean={}'.format(len(stats), baseline))
    return baseline

baseline_depth = calibrate_depth_baseline(seconds=3.0)


In [ ]:
def monitor_depth(label, seconds=MONITOR_SECONDS):
    print('[monitor:{}] start {}s'.format(label, seconds))
    start = time.time()
    previous = None
    latest = None
    frame = 0
    while time.time() - start < seconds:
        latest = depth.observe(region=DEPTH_REGION)
        if latest is None:
            print('[monitor:{}] no frame'.format(label))
            time.sleep(0.2)
            continue
        frame += 1
        delta = 'n/a' if previous is None else '{:+.3f}'.format(latest['mean'] - previous)
        print('[monitor:{} frame={:03d}] mean={:.3f} min={:.3f} max={:.3f} delta={}'.format(
            label, frame, latest['mean'], latest['min'], latest['max'], delta
        ))
        previous = latest['mean']
        time.sleep(MONITOR_INTERVAL_SECONDS)
    return latest


In [ ]:
def pickup_sequence():
    print('[flow] pickup_sequence')
    reset_arm()
    open_gripper()
    apply_pose('pre_grasp', PRE_GRASP_POSE, speed=ARM_SPEED)
    apply_pose('reach', REACH_POSE, speed=ARM_SPEED)
    close_gripper()
    apply_pose('lift', LIFT_POSE, speed=ARM_SPEED)

def confirm_pickup_success(before_mean, after_stats):
    if before_mean is None or after_stats is None:
        print('[confirm] insufficient depth data')
        return False
    delta = after_stats['mean'] - before_mean
    success = abs(delta) >= SUCCESS_DEPTH_DELTA
    print('[confirm] before={:.3f} after={:.3f} delta={:+.3f} threshold={:.3f} success={}'.format(
        before_mean, after_stats['mean'], delta, SUCCESS_DEPTH_DELTA, success
    ))
    return success


In [ ]:
# Main workflow cell. Manually place the can before running.
try:
    print('[flow] start manual-placement demo')
    hardcoded_base_move()

    print('[flow] observe before pickup')
    before = monitor_depth('before_pickup', seconds=5.0)
    before_mean = None if before is None else before['mean']

    pickup_sequence()

    print('[flow] monitor after lift for success confirmation')
    after = monitor_depth('after_lift', seconds=MONITOR_SECONDS)
    success = confirm_pickup_success(before_mean, after)
    print('[flow] success signal:', success)
finally:
    print('[flow] final reset')
    reset_arm()
    depth.stop()
    if robot is not None:
        robot.stop()
